# Study 02 - LQR, Value Function, And Riccati Solvers

This study deepens the same double-integrator example. The main idea is that the DARE matrix $P$ is not just a tuning artifact: in the unconstrained linear-quadratic case it is the value-function matrix.


In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)

from scipy.linalg import solve_discrete_are, solve_discrete_lyapunov

STUDY_DIR = Path('studies/study_02_lqr_value_iteration_policy_iteration')
OUTPUT_DIR = STUDY_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Model And Cost

The infinite-horizon cost is

$$J = \sum_{k=0}^{\infty} x_k^T Q x_k + u_k^T R u_k.$$

For a stabilizing feedback $u_k = -Kx_k$, the value is $V(x)=x^TPx$.


In [ ]:
dt = 0.1
A = np.array([[1.0, dt], [0.0, 1.0]])
B = np.array([[0.5 * dt**2], [dt]])
Q = np.diag([10.0, 1.0])
R = np.array([[0.2]])


## SciPy DARE Solution


In [ ]:
def gain_from_P(A, B, R, P):
    return np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)

P_dare = solve_discrete_are(A, B, Q, R)
K_dare = gain_from_P(A, B, R, P_dare)
print('P from scipy DARE =\n', P_dare)
print('K from scipy DARE =', K_dare)


## Riccati Fixed-Point Iteration

Value iteration repeatedly applies the Riccati update until $P$ stops changing.


In [ ]:
def riccati_fixed_point(A, B, Q, R, iterations=120):
    P = Q.copy()
    errors = []
    for _ in range(iterations):
        K = gain_from_P(A, B, R, P)
        P_next = Q + A.T @ P @ A - A.T @ P @ B @ K
        errors.append(float(np.linalg.norm(P_next - P)))
        P = P_next
    return P, gain_from_P(A, B, R, P), np.array(errors)

iterations = 60
# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.
P_vi, K_vi, vi_errors = riccati_fixed_point(A, B, Q, R, iterations=iterations)
print('P from Riccati fixed-point =\n', P_vi)
print('K from Riccati fixed-point =', K_vi)
print('P error vs DARE:', np.linalg.norm(P_vi - P_dare))


## Policy Iteration

Policy iteration alternates between evaluating a fixed policy and improving it.


In [ ]:
def policy_iteration(A, B, Q, R, K0, iterations=20):
    K = K0.copy()
    errors = []
    for _ in range(iterations):
        Acl = A - B @ K
        P = solve_discrete_lyapunov(Acl.T, Q + K.T @ R @ K)
        K_next = gain_from_P(A, B, R, P)
        errors.append(float(np.linalg.norm(K_next - K)))
        K = K_next
    return P, K, np.array(errors)

K0 = np.array([[0.5, 1.0]])
# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.
P_pi, K_pi, pi_errors = policy_iteration(A, B, Q, R, K0)
print('P from policy iteration =\n', P_pi)
print('K from policy iteration =', K_pi)
print('K error vs DARE:', np.linalg.norm(K_pi - K_dare))


## Comparison Plot


In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(vi_errors, label='Riccati fixed-point ||P_next - P||')
plt.semilogy(pi_errors, label='policy iteration ||K_next - K||')
plt.xlabel('iteration')
plt.ylabel('error')
plt.grid(True, alpha=0.25)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'riccati_policy_convergence.png', dpi=150)
plt.show()


## Student Questions

- Why does $P$ define the cost-to-go?
- Which method converges faster here?
- What happens if the initial policy is not stabilizing?
